In [37]:
# install dependencies ───────────────────────────
!pip install pymupdf langchain langchain-text-splitters \
             chromadb sentence-transformers groq tqdm python-dotenv \
             -q

In [4]:
#  mount Google Drive ─────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# your papers folder
PAPERS_DIR = "/content/drive/MyDrive/rag_responsible_ai_project/responsible_ai_papers"

Mounted at /content/drive


In [39]:
# imports ─────────────────────────────────────────
from google.colab import userdata
import fitz #pymupdf
import os, json, re, time
from pathlib import Path
from collections import Counter
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
from groq import Groq
import warnings
import os
import logging
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


CHUNKS_FILE = "/content/drive/MyDrive/rag_responsible_ai_project/chunks.json"

In [7]:
# Configs
PAPERS_DIR      = "/content/drive/MyDrive/rag_responsible_ai_project/responsible_ai_papers"
CHUNKS_FILE     = "/content/drive/MyDrive/rag_responsible_ai_project/chunks.json"
CHROMA_DIR      = "/content/drive/MyDrive/rag_responsible_ai_project/chroma_db"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
RERANK_MODEL    = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GROQ_MODEL      = "llama-3.1-8b-instant"
COLLECTION_NAME = "responsible_ai_papers"
CHUNK_SIZE      = 1000
CHUNK_OVERLAP   = 200
BATCH_SIZE      = 64
RETRIEVE_N      = 20
TOP_K           = 5
MAX_PER_PAPER   = 2

GROQ_API_KEY    = userdata.get('rag_api_key')


print("Config ready.")

Config ready.


In [8]:
# extract text from PDFs ─────────────────────────
def clean_text(text):
    """Remove common PDF junk — headers, footers, excessive whitespace."""
    # join hyphenated line breaks e.g. "discrimina-\ntory" → "discriminatory"
    text = re.sub(r'(\w+)-\s*\n\s*(\w+)', r'\1\2', text)
    # collapse multiple newlines
    text = re.sub(r'\n{3,}', '\n\n', text)
    # collapse multiple spaces
    text = re.sub(r' {2,}', ' ', text)
    # remove lines that are just numbers (page numbers)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.MULTILINE)
    # remove leading period leftover from sentence splitting
    text = re.sub(r'^\.\s*', '', text)
    # remove references section — everything after "References" or "Bibliography"
    text = re.sub(r'\n(References|Bibliography|REFERENCES)\n.*', '', text, flags=re.DOTALL)
    # remove lines shorter than 60 chars that repeat more than 3 times
    # these are usually headers/footers
    lines = text.split('\n')
    line_counts = Counter(lines)
    text = '\n'.join(
        line for line in lines
        if line_counts[line] <= 3 or len(line) > 60
    )
    return text.strip()

pdf_files = list(Path(PAPERS_DIR).glob("*.pdf"))
print(f"Found {len(pdf_files)} PDFs\n")

documents = []  # each item = {text, filename, title}

for pdf_path in pdf_files:
    try:
        doc  = fitz.open(str(pdf_path))
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()

        text = clean_text(text)

        if len(text) < 500:
            print(f"⚠ Skipping {pdf_path.name} — too little text extracted")
            continue

        documents.append({
            "filename": pdf_path.name,
            "title":    pdf_path.stem,  # filename without .pdf
            "text":     text,
        })
        print(f"✓ {pdf_path.name} — {len(text):,} characters")

    except Exception as e:
        print(f"✗ Failed: {pdf_path.name} — {e}")

print(f"\nSuccessfully extracted: {len(documents)} documents")

Found 20 PDFs

✓ 1910.10045v2.pdf — 26,708 characters
✓ 2305.02231v2.pdf — 146,848 characters
✓ 3531146.3533231.pdf — 62,598 characters
✓ Thinking responsibly about responsible AI and  the dark side  of AI.pdf — 15,373 characters
✓ 2101.02032v5.pdf — 103,972 characters
✓ 06c5e65f62e2264f3fb94c819ce1d2a7dde8.pdf — 78,663 characters
✓ 2205.07722v2.pdf — 47,809 characters
✓ 2311.14705v1.pdf — 87,629 characters
✓ Raso+(Updated+May+2nd).pdf — 63,342 characters
✓ 3544548.3581278.pdf — 118,626 characters
✓ s41598-025-25266-z.pdf — 50,950 characters
✓ ozturkcan-bozdağ-2025-responsible-ai-in-marketing-ai-booing-and-ai-washing-cycle-of-ai-mistrust.pdf — 67,927 characters
✓ s43681-025-00809-2.pdf — 62,442 characters
✓ e817149ac042ef2f6ad5ce40f6ad8230a97e.pdf — 17,584 characters
✓ 2506.09873v1.pdf — 64,980 characters
✓ 1-s2.0-S2666920X25000621-main.pdf — 72,614 characters
✓ 2504.16148v1.pdf — 141,032 characters
✓ 2503.09858v1.pdf — 66,579 characters
✓ 2502.18359v1.pdf — 158,191 characters
✓ 7df0c

In [9]:
#  chunk the text ──────────────────────────────────
# chunk_size: how many characters per chunk
# chunk_overlap: how many characters overlap between chunks
#   (overlap is important — it prevents losing context at boundaries)

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 200,
    separators = ["\n\n", ". ", "\n", " ", ""]
)

all_chunks = []

for doc in documents:
    chunks = splitter.split_text(doc["text"])

    for i, chunk_text in enumerate(chunks):
    # clean up any leading period/whitespace left over from sentence splitting
      chunk_text = re.sub(r'^[\.\s]+', '', chunk_text).strip()
      all_chunks.append({
          "chunk_id":  f"{doc['filename']}__chunk_{i:04d}",
          "filename":  doc["filename"],
          "title":     doc["title"],
          "chunk_idx": i,
          "text":      chunk_text,
      })

print(f"Total chunks: {len(all_chunks)}")
print(f"Average chunk size: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} characters")
print(f"Chunks per document (average): {len(all_chunks) // len(documents)}")


Total chunks: 1949
Average chunk size: 847 characters
Chunks per document (average): 97


In [10]:
# inspect a sample chunk ─────────────────────────


sample = all_chunks[21]  # pick chunk #10 as a sample
print("=== SAMPLE CHUNK ===")
print(f"From:  {sample['title']}")
print(f"Chunk: {sample['chunk_idx']}")
print(f"Size:  {len(sample['text'])} characters")
print(f"\n{sample['text']}")

=== SAMPLE CHUNK ===
From:  1910.10045v2
Chunk: 21
Size:  291 characters

This notion of model comprehensibility stems from the postulates of Michalski [22], which stated that
“the results of computer induction should be symbolic descriptions of given entities, semantically and
structurally similar to those a human expert might produce observing the same entities


In [11]:
#  save chunks ─────────────────────────────────────
with open(CHUNKS_FILE, "w") as f:
    json.dump(all_chunks, f, indent=2)

print(f"\nSaved {len(all_chunks)} chunks to {CHUNKS_FILE}")


Saved 1949 chunks to /content/drive/MyDrive/rag_responsible_ai_project/chunks.json


In [12]:
#  load chunks ─────────────────────────────────────

with open(CHUNKS_FILE, "r") as f:
    all_chunks = json.load(f)

print(f"Loaded {len(all_chunks)} chunks from {CHUNKS_FILE}")

Loaded 1949 chunks from /content/drive/MyDrive/rag_responsible_ai_project/chunks.json


In [13]:
# ⚠ CAUTION — only run manually when need to wipe and rebuild
# delete existing collection to force re-embedding
#try:
#    client = chromadb.PersistentClient(path=CHROMA_DIR)
#    client.delete_collection(COLLECTION_NAME)
#    print("Old collection deleted. Ready to re-embed.")
#except Exception as e:
#    print(f"Nothing to delete: {e}")

In [14]:
os.makedirs(CHROMA_DIR, exist_ok=True)

client     = chromadb.PersistentClient(path=CHROMA_DIR)


collection = client.get_or_create_collection(
    name     = COLLECTION_NAME,
    metadata = {"hnsw:space": "cosine"}

)
print(f"Collection '{COLLECTION_NAME}' ready.")
print(f"Currently contains {collection.count()} embeddings.")

Collection 'responsible_ai_papers' ready.
Currently contains 1949 embeddings.


In [41]:
#  load embedding model ───────────────────────────

print(f"Loading embedding model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)
print("Model loaded.")

Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded.


In [16]:
#  embed and store chunks (scalable) ──────────────

# get all chunk_ids already stored in ChromaDB
existing_ids = set(collection.get(include=[])["ids"])
print(f"Already embedded: {len(existing_ids)} chunks")

# filter to only chunks that aren't embedded yet
new_chunks = [c for c in all_chunks if c["chunk_id"] not in existing_ids]
print(f"New chunks to embed: {len(new_chunks)}")

if len(new_chunks) == 0:
    print("Nothing to do — all chunks already embedded!")
else:

    total_batches = (len(new_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

    for i in tqdm(range(0, len(new_chunks), BATCH_SIZE),
                  desc="Embedding chunks", total=total_batches):

        batch = new_chunks[i : i + BATCH_SIZE]


        texts = [c["text"] for c in batch]


        embeddings = model.encode(texts, show_progress_bar=False).tolist()

        # store in ChromaDB — three things per chunk:
        # 1. ids       — unique identifier for each chunk
        # 2. embeddings — the vectors we just computed
        # 3. documents  — the original text (for retrieval later)
        # 4. metadatas  — title, filename etc (for citations later)
        collection.add(
            ids        = [c["chunk_id"]  for c in batch],
            embeddings = embeddings,
            documents  = [c["text"]      for c in batch],
            metadatas  = [
                {
                    "title":     c["title"],
                    "filename":  c["filename"],
                    "chunk_idx": c["chunk_idx"],
                }
                for c in batch
            ]
        )

    print(f"\nDone. Total embeddings in DB: {collection.count()}")

Already embedded: 1949 chunks
New chunks to embed: 0
Nothing to do — all chunks already embedded!


In [17]:
#  verify with a test query ───────────────────────


test_question = "What are the main challenges of responsible AI?"

# embed the question the same way we embedded the chunks
question_embedding = model.encode(test_question).tolist()

# query ChromaDB for top 3 most similar chunks
results = collection.query(
    query_embeddings = [question_embedding],
    n_results        = 3,
    include          = ["documents", "metadatas", "distances"]
)

print(f"Test query: '{test_question}'\n")
print("Top 3 results:\n")

for i, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
), 1):
    # distance is cosine distance — lower = more similar
    # we convert to similarity score: 1 - distance
    similarity = round(1 - dist, 3)
    print(f"Result {i} — similarity: {similarity}")
    print(f"From: {meta['title']}")
    print(f"Text: {doc[:200]}...")
    print()

Test query: 'What are the main challenges of responsible AI?'

Top 3 results:

Result 1 — similarity: 0.689
From: s43681-025-00809-2
Text: Such conditions prompt a 
critical question: Are the pillars of Responsible AI built on 
solid foundations, or are they, in practice, constructed on 
sand?
At its core, Responsible AI aspires to mitig...

Result 2 — similarity: 0.679
From: Raso+(Updated+May+2nd)
Text: A tension is thus built into the Declaration. Even as it divides artificial from human intelligence to explain how 
AI systems operate and to compartmentalize the system’s components, its 
very terms ...

Result 3 — similarity: 0.679
From: Thinking responsibly about responsible AI and  the dark side  of AI
Text: We start by discussing the phenom­
enon of responsible AI in the next section, its origins, 
and the current discussion around the notion. The 
notion of responsible AI is then decomposed and used 
as...



In [40]:
# ── CELL 3 — load models and connect to ChromaDB ────────────


print("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL)

# Load the cross-encoder for reranking

print("Loading reranking model...")
reranker = CrossEncoder(RERANK_MODEL)

# Connect persistent ChromaDB
client     = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection(COLLECTION_NAME)

print(f"Connected to ChromaDB — {collection.count()} chunks indexed.")
print("All models loaded.")

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading reranking model...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Connected to ChromaDB — 1949 chunks indexed.
All models loaded.


In [19]:
# retrieval function ──────────────────────────────
# Takes a question, returns top RETRIEVE_N chunks from ChromaDB

def retrieve(question, n=RETRIEVE_N):
    """
    Embed the question and find the most similar chunks
    in ChromaDB using cosine similarity.
    Returns a list of dicts with text, title, filename, score.
    """
    # embed the question — same model as chunks
    question_embedding = embedder.encode(question).tolist()

    # query ChromaDB
    results = collection.query(
        query_embeddings = [question_embedding],
        n_results        = n,
        include          = ["documents", "metadatas", "distances"]
    )

    # package results into clean dicts
    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append({
            "text":       doc,
            "title":      meta["title"],
            "filename":   meta["filename"],
            "similarity": round(1 - dist, 3),
        })

    return chunks

In [20]:
#  reranking function ──────────────────────────────
# Takes the retrieved chunks and reranks them using a
# cross-encoder that scores (question, chunk) pairs together

def rerank(question, chunks, top_k=TOP_K, max_per_paper=MAX_PER_PAPER):
    """
    Rerank chunks using a cross-encoder model.
    Also enforces diversity — max_per_paper chunks per paper.
    Returns top_k most relevant chunks.
    """
    if not chunks:
        return []

    # create (question, chunk_text) pairs for the cross-encoder
    pairs = [(question, c["text"]) for c in chunks]

    # cross-encoder scores each pair — higher = more relevant
    # unlike embedding similarity, this score is not bounded 0-1
    # it's a raw relevance score, higher is better
    scores = reranker.predict(pairs)

    # attach scores to chunks
    for chunk, score in zip(chunks, scores):
        chunk["rerank_score"] = round(float(score), 3)

    # sort by rerank score descending
    ranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)

    # enforce diversity — max max_per_paper chunks per paper
    # without this, one highly relevant paper could fill all top_k spots
    paper_counts = {}
    diverse      = []

    for chunk in ranked:
        paper = chunk["filename"]
        count = paper_counts.get(paper, 0)
        if count < max_per_paper:
            diverse.append(chunk)
            paper_counts[paper] = count + 1
        if len(diverse) == top_k:
            break

    return diverse

In [21]:
#  generation function ────────────────────────────
# Takes the top reranked chunks and sends them to the LLM
# with the question to generate a grounded answer with citations

def generate(question, chunks, groq_api_key=GROQ_API_KEY):
    """
    Send retrieved chunks as context to the LLM.
    The LLM is instructed to answer ONLY from the provided context
    and cite the papers it uses.
    """
    # build the context string from chunks
    # each chunk is labeled with its source paper
    context_parts = []
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[Source {i}: {chunk['title']}]\n{chunk['text']}"
        )
    context = "\n\n---\n\n".join(context_parts)

    # the system prompt is critical in RAG
    # it tells the LLM to:
    # 1. answer ONLY from the provided context (prevents hallucination)
    # 2. cite sources by number (enables traceability)
    # 3. admit when it doesn't know (prevents making things up)
    system_prompt = """You are a research assistant specializing in Responsible AI.
Answer the user's question using ONLY the provided context from research papers.
Always cite your sources using [Source N] notation.
If the context does not contain enough information to answer the question, say so clearly.
Do not use any knowledge outside of the provided context."""

    user_prompt = f"""Context from research papers:

{context}

Question: {question}

Answer with citations:"""

    # call Groq API
    client   = Groq(api_key=groq_api_key)
    response = client.chat.completions.create(
        model    = GROQ_MODEL,
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature = 0.1,  # low temperature = more factual, less creative
        max_tokens  = 1024,
    )

    return response.choices[0].message.content

In [22]:
#  full RAG pipeline function ─────────────────────
# Combines retrieve + rerank + generate into one clean function
# This is what Streamlit app will call
def ask(question, verbose=True):
    """
    Full RAG pipeline:
    1. Retrieve top 20 chunks from ChromaDB
    2. Rerank to top 5 with diversity enforcement
    3. Generate answer with citations
    """
    if verbose:
        print(f"Question: {question}\n")

    # step 1: retrieve
    retrieved = retrieve(question)
    if verbose:
        print(f"Retrieved {len(retrieved)} chunks.")

    # step 2: rerank
    reranked = rerank(question, retrieved)
    if verbose:
        print(f"Reranked to {len(reranked)} chunks.")
        print("\nSources used:")
        for i, c in enumerate(reranked, 1):
            print(f"  [{i}] {c['title']} (rerank score: {c['rerank_score']})")

    # step 3: generate
    if verbose:
        print("\nGenerating answer...\n")
    answer = generate(question, reranked)

    if verbose:
        print("=" * 60)
        print(answer)
        print("=" * 60)

    return {
        "question": question,
        "answer":   answer,
        "sources":  reranked,
    }

In [23]:
# ── CELL 8 — test the full pipeline ─────────────────────────
# Test with a few real questions before building the UI

test_questions = [
    "What are the main challenges of responsible AI?",
    "How does bias in AI systems affect hiring decisions?",
    "What is the relationship between explainability and accountability in AI?",
]

for question in test_questions:
    result = ask(question)
    print("\n")

Question: What are the main challenges of responsible AI?

Retrieved 20 chunks.
Reranked to 5 chunks.

Sources used:
  [1] s43681-025-00809-2 (rerank score: 6.241)
  [2] s43681-025-00809-2 (rerank score: 5.956)
  [3] 7df0c76dd9ae6af09f8fc35de4bd59f8b97f (rerank score: 5.801)
  [4] Thinking responsibly about responsible AI and  the dark side  of AI (rerank score: 3.23)
  [5] e817149ac042ef2f6ad5ce40f6ad8230a97e (rerank score: 3.186)

Generating answer...

According to the provided context, the main challenges of responsible AI include:

1. Systemic issues that demand a fundamental shift in how responsibility in AI is understood and enacted [Source 2: s43681-025-00809-2].
2. Recurring structural tensions across the foundational pillars of responsible AI, including fairness, transparency, accountability, privacy, safety, and value alignment [Source 2: s43681-025-00809-2].
3. Challenges in AI safety, such as data bias, lack of transparency, copyright concerns, and accountability challenges

In [24]:
# ── CELL 9 — stress tests ────────────────────────────────────

stress_tests = [
    # hallucination tests
    "What did the EU AI Act specifically mandate in Article 13?",
    "What did Geoffrey Hinton say about responsible AI in 2024?",
    "What are the exact penalty amounts for violating AI fairness regulations in Germany?",

    # multi-hop reasoning
    "How does the lack of explainability in deep learning models create accountability gaps in high-stakes decisions?",
    "What is the relationship between data governance and algorithmic fairness?",
    "How do power imbalances between AI developers and affected communities undermine responsible AI principles?",

    # vague questions
    "Is AI good or bad?",
    "What should companies do about AI?",
    "How do we fix bias?",

    # out of scope
    "What is the best programming language for building AI systems?",
    "What is the capital of France?",

    # conflicting information
    "Is transparency always beneficial in AI systems?",
    "Can AI ever be truly fair?",
    "Is explainability sufficient for trustworthy AI?",
]

for question in stress_tests:
    print(f"\n{'='*60}")
    result = ask(question, verbose=False)  # verbose=False for cleaner output
    print(f"Q: {question}")
    print(f"\nSources: {[c['title'] for c in result['sources']]}")
    print(f"\nA: {result['answer']}")


Q: What did the EU AI Act specifically mandate in Article 13?

Sources: ['2305.02231v2', '2506.09873v1', '2305.02231v2', '2506.09873v1', 'e817149ac042ef2f6ad5ce40f6ad8230a97e']

A: Unfortunately, the provided context does not contain enough information to answer the question about what the EU AI Act specifically mandated in Article 13.

Q: What did Geoffrey Hinton say about responsible AI in 2024?

Sources: ['2305.02231v2', 'Thinking responsibly about responsible AI and  the dark side  of AI', 'Raso+(Updated+May+2nd)', 'Raso+(Updated+May+2nd)', 'Thinking responsibly about responsible AI and  the dark side  of AI']

A: There is no information about what Geoffrey Hinton said about responsible AI in 2024 in the provided context. The information about Geoffrey Hinton is from 2023, where he stated "We need to find a way to control artificial intelligence before it’s too late" [Source 1: 2305.02231v2, 22].

Q: What are the exact penalty amounts for violating AI fairness regulations in Germa

# Pushing On Github

In [25]:
# ── configure git ────────────────────────────────────────────
!git config --global user.email "arabidona13@gmail.com"
!git config --global user.name "Donna737"

In [27]:
# ── clone your empty repo ────────────────────────────────────
import os
os.chdir("/content")

# replace with your actual GitHub username and repo name
!git clone https://github.com/Donna737/responsible-ai-rag
os.chdir("/content/responsible-ai-rag")

Cloning into 'responsible-ai-rag'...
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 5 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (5/5), done.


In [29]:
import subprocess
result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-name", "*.ipynb"],
    capture_output=True, text=True
)
print(result.stdout)

/content/drive/MyDrive/Colab Notebooks/RAG_project.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/tutorial08_MountainCar_DQN_template (1).ipynb
/content/drive/MyDrive/Colab Notebooks/tutorial08_MountainCar_DQN_template.ipynb
/content/drive/MyDrive/Colab Notebooks/ex_09_MountainCar_PG_temlate.ipynb
/content/drive/MyDrive/Colab Notebooks/One.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/three.ipynb
/content/drive/MyDrive/Colab Notebooks/four.ipynb
/content/drive/MyDrive/Colab Notebooks/five.ipynb
/content/drive/MyDrive/Colab Notebooks/what_happened_to_hyperparameters.ipynb
/content/drive/MyDrive/Colab Notebooks/transformers.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/GTEA_this_time.ipynb
/content/drive/MyDrive/Colab Notebooks/hacking_stuff.ipynb
/content/drive/MyDrive/Colab Notebooks/mastering_the_chaos.ipynb
/content/drive/MyD

In [44]:
# copy notebook into the repo folder
import shutil
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/RAG_project.ipynb",
    "/content/responsible-ai-rag/RAG_project.ipynb"
)
print("Copied.")

Copied.


In [47]:
os.chdir("/content/responsible-ai-rag")
!git add RAG_project.ipynb
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [46]:
!git add .
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [33]:
!git commit -m "Days 1-4: PDF extraction, chunking, embedding, reranking, generation"

[main c841925] Days 1-4: PDF extraction, chunking, embedding, reranking, generation
 1 file changed, 1 insertion(+)
 create mode 100644 RAG_project.ipynb


In [34]:
# replace with your actual values
GITHUB_TOKEN    = userdata.get("GITHUB_TOKEN")
GITHUB_USERNAME = "Donna737"
REPO_NAME       = "responsible-ai-rag"

!git remote set-url origin https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git
!git push origin main

Enumerating objects: 4, done.
Counting objects: 100% (4/4), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 32.05 KiB | 5.34 MiB/s, done.
Total 3 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/Donna737/responsible-ai-rag.git
   cab0ef8..c841925  main -> main


In [35]:
import json

notebook_path = "/content/drive/MyDrive/Colab Notebooks/RAG_project.ipynb"

with open(notebook_path, "r") as f:
    nb = json.load(f)

# fix the widgets metadata issue
if "widgets" in nb.get("metadata", {}):
    widgets = nb["metadata"]["widgets"]
    for mime_type, widget_data in widgets.items():
        if "state" not in widget_data:
            widget_data["state"] = {}

with open(notebook_path, "w") as f:
    json.dump(nb, f, indent=1)

print("Fixed.")

Fixed.


In [36]:
import shutil, os
shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/RAG_project.ipynb",
    "/content/responsible-ai-rag/RAG_project.ipynb"
)
os.chdir("/content/responsible-ai-rag")
!git add RAG_project.ipynb
!git commit -m "fix notebook widget metadata"
!git push origin main

[main abc8391] fix notebook widget metadata
 1 file changed, 9268 insertions(+), 1 deletion(-)
 rewrite RAG_project.ipynb (98%)
Enumerating objects: 5, done.
Counting objects: 100% (5/5), done.
Delta compression using up to 2 threads
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 34.61 KiB | 3.84 MiB/s, done.
Total 3 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Donna737/responsible-ai-rag.git
   c841925..abc8391  main -> main
